# Lab 2 – Chunking

**Câu hỏi:** cắt tài liệu thế nào để truy xuất đúng điều khoản? So sánh trên cùng output Docling của Lab 1, cùng embedding (BGE-M3 dense), cùng golden set 20 câu:

| Chiến lược | Cách cắt | Thứ đem đi embed |
|---|---|---|
| `fixed` | cửa sổ 512 token, chồng lấp 64 | nội dung |
| `recursive` | `\n\n` → `\n` → câu → từ, ≤ 512 token, chồng lấp 64 | nội dung |
| `structure` | theo Chương/Phần › Điều › mục; bảng gom theo nhóm từ tables JSON; bỏ mục lục, biểu mẫu, trang tóm tắt trùng | breadcrumb + nội dung |
| `structure_nobc` | như `structure` | **chỉ** nội dung (ablation: tách tác dụng của breadcrumb) |
| `structure+ctx` (bonus) | như `structure`, riêng AVSK | câu ngữ cảnh do LLM viết + breadcrumb + nội dung |

**Chấm điểm:** chunk đúng = đúng `doc_id` và chứa đủ đoạn trích `must_contain` của bằng chứng (không phụ thuộc cách cắt). Truy xuất trên **toàn corpus** (6 tài liệu), không lọc theo sản phẩm.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 220)

from ragbv.config import load_manifest, REPORTS
from ragbv.parsing import load_pages
from ragbv import chunking as C, embedding as EMB, metrics as M
from ragbv.golden import load_golden, contains_all

docs = load_manifest()
pages = {d: load_pages(d, "docling") for d in docs}
golden = [g for g in load_golden() if g.in_scope]
print(len(golden), "câu có bằng chứng")

20 câu có bằng chứng


## 1. Tạo chunk

In [2]:
chunks = {name: [c for d, m in docs.items() for c in fn(m, pages[d])] for name, fn in C.STRATEGIES.items()}
chunks["structure_nobc"] = [C.Chunk(**{**c.__dict__, "strategy": "structure_nobc", "embed_text": c.text})
                            for c in chunks["structure"]]
for name, cs in chunks.items():
    C.save_chunks(name, cs)

def stats(cs):
    t = np.array([c.n_tokens for c in cs])
    return {"chunks": len(cs), "tok_mean": round(t.mean()), "tok_p50": int(np.median(t)), "tok_max": int(t.max()),
            "tok_total": int(t.sum()), "table_chunks": sum(c.kind == "table" for c in cs)}
pd.DataFrame({k: stats(v) for k, v in chunks.items()}).T

C:\Users\Legion\Desktop\Job\Paraline\Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (21231 > 8192). Running this sequence through the model will result in indexing errors


,chunks,tok_mean,tok_p50,tok_max,tok_total,table_chunks
fixed,197,505,512,513,99425,0
recursive,226,407,453,511,92033,0
structure,402,239,215,483,96052,101
structure_nobc,402,239,215,483,96052,101


In [3]:
# Trần trên: bằng chứng có nằm trọn trong ít nhất 1 chunk không? (nếu không, chiến lược đó không bao giờ tìm được)
ceiling = {name: sum(any(c.doc_id == ev.doc_id and contains_all(c.text, ev.must_contain) for c in cs)
                     for g in golden for ev in g.evidence) for name, cs in chunks.items()}
total_ev = sum(len(g.evidence) for g in golden)
{k: f"{v}/{total_ev}" for k, v in ceiling.items()}

{'fixed': '28/28',
 'recursive': '28/28',
 'structure': '28/28',
 'structure_nobc': '28/28'}

### Cùng một điều khoản, 3 cách cắt
Điều 8.1 AVSK (thời gian chờ thai sản 270 ngày, G11): chunk chứa bằng chứng trong từng chiến lược.

In [4]:
ev = next(g for g in golden if g.id == "G11").evidence[0]
for name in ["fixed", "recursive", "structure"]:
    c = next(c for c in chunks[name] if c.doc_id == ev.doc_id and contains_all(c.text, ev.must_contain))
    print(f"===== {name} | {c.n_tokens} token | trang {c.page_labels} =====")
    print(c.embed_text[:900], "\n")

===== fixed | 512 token | trang ['15', '16'] =====
bảo hiểm nào theo Sản phẩm bổ trợ này, Bên mua bảo hiểm sẽ được giảm phí cho một năm tái tục tiếp theo.

Tỉ lệ giảm phí bảo hiểm tái tục cho từng Quyền lợi bảo hiểm nêu tại Khoản 6.1, 6.2, 6.3 Điều 6 như sau:

| Điều kiện xét giảm phí | Tỉ lệ giảm phí (% phí bảo hiểm của Quyền lợi bảo hiểm nêu tại Khoản 6.1, 6.2, 6.3 Điều 6 cho một năm tái tục tiếp theo) |
|-------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------|
| Không phát sinh yêu cầu chi trả theo Quyền lợi điều trị tương ứng trong 01 năm tham gia liền trước thời điểm xem xét | 5% |
| Không phát sinh yêu cầu chi trả theo Quyền lợi điều trị tương ứng trong 02 năm tham gia liên tục liền trước thời điểm xem xét | 10% |
| Không phát sinh yêu cầu 

===== recursive | 367 token | trang ['1'] ====

## 2. Embed và truy xuất (BGE-M3 dense, cosine, toàn corpus)

In [5]:
queries = [g.question for g in golden]
qv = EMB.encode(queries)
vecs = {name: EMB.encode([c.embed_text for c in cs]) for name, cs in chunks.items()}

def evaluate(name, cs, dv, items=golden, k=10):
    top = EMB.search(EMB.encode([g.question for g in items]), dv, k)
    return pd.DataFrame([{"strategy": name, **M.score_item(g, [cs[i] for i in row])} for g, row in zip(items, top)])

results = pd.concat([evaluate(n, chunks[n], vecs[n]) for n in chunks])
summary = results.groupby("strategy")[["hit@1", "hit@3", "hit@5", "hit@10", "recall@5", "recall@10", "mrr"]].mean().round(3)
summary.loc[["fixed", "recursive", "structure_nobc", "structure"]]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 1604.55it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   1%|▏         | 5/391 [00:03<04:49,  1.33it/s]

Loading weights: 100%|██████████| 391/391 [00:03<00:00, 103.00it/s]

pre tokenize:   0%|          | 0/7 [00:00<?, ?it/s]

... (5 outputs left out)

Inference Embeddings:  57%|█████▋    | 4/7 [00:00<00:00,  5.66it/s]

Inference Embeddings:  86%|████████▌ | 6/7 [00:00<00:00,  8.17it/s]

Inference Embeddings: 100%|██████████| 7/7 [00:00<00:00,  7.56it/s]

,hit@1,hit@3,hit@5,hit@10,recall@5,recall@10,mrr
strategy,,,,,,,
fixed,0.25,0.65,0.70,0.85,0.633,0.808,0.452
recursive,0.45,0.65,0.75,0.80,0.708,0.800,0.578
structure_nobc,0.35,0.60,0.70,0.85,0.650,0.808,0.511
structure,0.60,0.70,0.85,0.90,0.783,0.858,0.691


### Theo loại câu hỏi (recall@5)

In [6]:
results.pivot_table(index="type", columns="strategy", values="recall@5", aggfunc="mean").round(2)[
    ["fixed", "recursive", "structure_nobc", "structure"]]

strategy,fixed,recursive,structure_nobc,structure
type,,,,
comparison,0.58,0.58,0.25,0.58
exclusion,0.00,0.00,0.00,1.00
fact,0.83,0.83,0.83,0.83
multi_condition,1.00,1.00,1.00,1.00
no_diacritics,0.25,0.50,0.25,0.25
table_lookup,0.40,0.60,0.60,0.80
versioning,1.00,1.00,1.00,1.00


### Từng câu: hạng của bằng chứng (None = ngoài top-10)

In [7]:
rank_table = results.pivot(index="id", columns="strategy", values="ranks")[["fixed", "recursive", "structure_nobc", "structure"]]
rank_table.join(pd.Series({g.id: g.question[:70] for g in golden}, name="question"))

,fixed,recursive,structure_nobc,structure,question
id,,,,,
G01,[None],[None],[None],[1],Phí bảo hiểm điều khoản chính (điều trị nội trú) gói Bạc của Bảo Việt
G02,[8],[2],[1],[1],Tổng hạn mức quyền lợi bảo hiểm chính của gói Vàng trong Bảo Việt An G
G03,[10],[None],[None],[None],"Với quyền lợi thai sản của Manulife Chăm Sóc Sức Khỏe Nâng Cao, chi ph"
G04,[1],[1],[1],[1],Bảo hiểm thai sản của Bảo Việt An Gia áp dụng cho ai và sau bao lâu th
G05,[2],[1],[1],[1],Trẻ em từ 15 ngày tuổi đến 3 tuổi tham gia Bảo Việt An Gia được chi tr
G06,"[2, 2]","[1, 1]","[3, 3]","[1, 1]",Thời gian chờ đối với bệnh có sẵn và bệnh đặc biệt trong quy tắc Bảo V
G07,[None],[1],[1],[1],Bảo Việt Tâm An nhận bảo hiểm cho người ở độ tuổi nào?
G08,[3],[4],[2],[1],Chương trình Lộc của Bảo Việt Tâm An trợ cấp bệnh viện công bao nhiêu?
G09,"[4, 1]","[1, 1]","[1, 1]","[1, 1]",Hồ sơ yêu cầu bồi thường Bảo Việt Tâm An phải nộp trong thời hạn nào v


## 3. Phân tích lỗi: câu `structure` trượt top-5

In [8]:
top_s = EMB.search(qv, vecs["structure"], 5)
for g, row in zip(golden, top_s):
    r = results[(results.strategy == "structure") & (results.id == g.id)].iloc[0]
    if r["recall@5"] < 1:
        print(f"##### {g.id} ({g.type}) recall@5={r['recall@5']:.2f} ranks={r['ranks']} | {g.question}")
        for i in row[:3]:
            c = chunks["structure"][i]
            print(f"   -> [{c.doc_id} tr.{c.page_labels}] {c.embed_text[:160]!r}")

##### G03 (table_lookup) recall@5=0.00 ranks=[None] | Với quyền lợi thai sản của Manulife Chăm Sóc Sức Khỏe Nâng Cao, chi phí sinh được chi trả như thế nào?
   -> [manulife-cssknc-2026 tr.['4']] '[Chăm Sóc Sức Khỏe Nâng Cao › Chương I. QUYỀN LỢI BẢO HIỂM › ĐIỀU 1. QUYỀN LỢI BẢO HIỂM]\nTheo đó:\n- Quyền lợi Điều Trị Nội Trú là quyền lợi bảo hiểm mặc định ch'
   -> [manulife-cssknc-2026 tr.['13']] '[Chăm Sóc Sức Khỏe Nâng Cao › Chương III. GIẢI QUYẾT QUYỀN LỢI BẢO HIỂM › Điều 6. GIẢI QUYẾT QUYỀN LỢI BẢO HIỂM]\n6.1. Người nhận quyền lợi bảo hiểm\nTrừ trường h'
   -> [manulife-cssknc-2026 tr.['22']] '[Chăm Sóc Sức Khỏe Nâng Cao › Chương VII. CÁC THUẬT NGỮ CẦN BIẾT]\nBảng (col0, col1)\n- 17. Chi Phí Xe Cứu Thương — col1: Là chi phí vận chuyển cấp cứu bằng xe cứ'
##### G11 (fact) recall@5=0.00 ranks=[6] | Thời gian chờ của quyền lợi thai sản trong sản phẩm An Vui Sống Khỏe là bao lâu?
   -> [bvl-avsk-2022 tr.['14']] '[An Vui Sống Khỏe › CHƯƠNG II: QUYỀN LỢI BẢO HIỂM › Điều 6: Quyền lợi bảo h

## 4. Bonus – Contextual Retrieval cho An Vui Sống Khỏe
LLM (`LLM_MODEL` trong `.env`) đọc toàn văn AVSK và viết 1–2 câu ngữ cảnh cho từng chunk `structure` của AVSK; câu ngữ cảnh được gắn trước breadcrumb khi embed. Các tài liệu khác giữ nguyên. Chấm trên các câu có bằng chứng thuộc AVSK.

In [9]:
from ragbv.contextual import contextualize
avsk_text = "\n\n".join(p.text for p in pages["bvl-avsk-2022"] if p.page_label not in ("1", "2"))
avsk = [c for c in chunks["structure"] if c.doc_id == "bvl-avsk-2022"]
avsk_ctx, cost = contextualize(avsk, avsk_text)
print(cost)
ctx_map = {c.chunk_id: c for c in avsk_ctx}
chunks["structure+ctx"] = [ctx_map.get(c.chunk_id, c) for c in chunks["structure"]]
C.save_chunks("structure+ctx", chunks["structure+ctx"])
vecs["structure+ctx"] = EMB.encode([c.embed_text for c in chunks["structure+ctx"]])
for c in avsk_ctx[:3]:
    print("--", c.meta["context"])

{'calls': 0, 'cached_calls': 58, 'prompt_tokens': 0, 'cached_prompt_tokens': 0, 'completion_tokens': 0}


pre tokenize:   0%|          | 0/4 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 254.54it/s]

Inference Embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

Inference Embeddings:  25%|██▌       | 1/4 [00:00<00:01,  1.60it/s]

Inference Embeddings:  50%|█████     | 2/4 [00:00<00:00,  2.86it/s]

Inference Embeddings:  75%|███████▌  | 3/4 [00:00<00:00,  4.13it/s]

Inference Embeddings: 100%|██████████| 4/4 [00:00<00:00,  4.25it/s]

-- Đoạn trích thuộc Quy tắc, Điều khoản Sản phẩm bổ trợ bảo hiểm sức khỏe của Bảo Việt Nhân thọ, được phê chuẩn theo Công văn số 2217/BTC-QLBH ngày 08/03/2022 của Bộ Tài chính, nằm trong phần mở đầu giới thiệu chung về sản phẩm và công ty.
-- Đoạn trích thuộc Quy tắc, Điều khoản Sản phẩm bổ trợ của Bảo Việt Nhân thọ, nằm trong Chương I, Điều 1, giới thiệu các định nghĩa và giải thích liên quan đến Người được bảo hiểm, Tuổi được bảo hiểm và các thuật ngữ quan trọng trong sản phẩm bổ trợ bảo hiểm y tế.
-- Đoạn trích thuộc Quy tắc, Điều khoản Sản phẩm bổ trợ bảo hiểm sức khỏe của Bảo Việt Nhân thọ, nằm trong Chương I, Điều 1, phần định nghĩa và giải thích về các thuật ngữ như "Tai nạn", "Bác sĩ" và "Cơ sở y tế" trong phạm vi bảo hiểm.


In [10]:
avsk_items = [g for g in golden if any(e.doc_id == "bvl-avsk-2022" for e in g.evidence)]
ctx_res = pd.concat([evaluate(n, chunks[n], vecs[n], avsk_items) for n in ["structure", "structure+ctx"]])
print([g.id for g in avsk_items])
ctx_res.groupby("strategy")[["hit@1", "hit@3", "hit@5", "recall@5", "mrr"]].mean().round(3)

['G11', 'G12', 'G13', 'G14', 'G15', 'G16']


,hit@1,hit@3,hit@5,recall@5,mrr
strategy,,,,,
structure,0.5,0.667,0.833,0.778,0.653
structure+ctx,0.5,0.833,1.000,0.944,0.708


In [11]:
ctx_res.pivot(index="id", columns="strategy", values="ranks")

strategy,structure,structure+ctx
id,,
G11,[6],[2]
G12,"[2, 1]","[1, 2]"
G13,[1],[1]
G14,[1],[1]
G15,[2],[2]
G16,"[5, 4, None]","[5, 4, None]"


In [12]:
# Toàn bộ golden set với structure+ctx (các tài liệu khác không đổi)
full_ctx = evaluate("structure+ctx", chunks["structure+ctx"], vecs["structure+ctx"])
pd.concat([summary.loc[["structure"]], full_ctx[["hit@1", "hit@3", "hit@5", "hit@10", "recall@5", "recall@10", "mrr"]].mean().round(3).rename("structure+ctx").to_frame().T])

,hit@1,hit@3,hit@5,hit@10,recall@5,recall@10,mrr
structure,0.6,0.70,0.85,0.9,0.783,0.858,0.691
structure+ctx,0.6,0.75,0.85,0.9,0.808,0.858,0.704


## 5. Lưu kết quả

In [13]:
REPORTS.mkdir(exist_ok=True)
out = {"chunk_stats": {k: stats(v) for k, v in chunks.items()}, "ceiling": ceiling, "total_evidence": total_ev,
       "summary": summary.reset_index().to_dict("records"),
       "by_type": results.pivot_table(index="type", columns="strategy", values="recall@5", aggfunc="mean").round(3).reset_index().to_dict("records"),
       "per_question": results.drop(columns=["ranks"]).assign(ranks=results["ranks"].astype(str)).to_dict("records"),
       "contextual": {"cost": cost, "avsk_items": [g.id for g in avsk_items],
                      "summary": ctx_res.groupby("strategy")[["hit@1", "hit@3", "hit@5", "recall@5", "mrr"]].mean().round(3).reset_index().to_dict("records"),
                      "full": full_ctx[["hit@1", "hit@3", "hit@5", "hit@10", "recall@5", "recall@10", "mrr"]].mean().round(3).to_dict()}}
(REPORTS / "lab2_results.json").write_text(json.dumps(out, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("saved")

saved
